# Kronos-TH Backtest Re-run (GPU)

GPU + Internet ON. Precompute is the bottleneck (~28 hrs/year on T4).
Use MODE to split across multiple Kaggle sessions:

| MODE | What it does | Est. time |
|------|-------------|-----------|
| `full` | precompute + walkforward (one-shot) | ~28 hrs — will timeout |
| `precompute` | download + precompute + save cache to Dataset | ~12 hrs — safe |
| `walkforward` | load cache from Dataset + walkforward + save | ~1 hr |

**Workflow:**
1. Set `MODE = "precompute"` → Run All (~12 hrs, cache saved before timeout safeguard)
2. Set `MODE = "precompute"` again → resumes where it left off (~12 hrs)
3. Repeat until precompute finishes
4. Set `MODE = "walkforward"` → Run All (~1 hr) → final results saved

The forecast cache is uploaded to `{KAGGLE_USERNAME}/kronos-forecast-cache-{YEAR}` Dataset after each precompute run.

In [ ]:
import os, sys, subprocess, json, time, shutil
from pathlib import Path

# -- Parameters ----------------------------------------------------------------
YEAR = "2023"
MODE = "precompute"  # "precompute" | "walkforward" | "full"
# ------------------------------------------------------------------------------
print(f"YEAR={YEAR} MODE={MODE}")

# -- Clone repos ---------------------------------------------------------------
REPO_URL = "https://github.com/Yutwizard/Kronos_Thai_Retail.git"
REPO_DIR = "/kaggle/working/Kronos_Thai_Retail"
if not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], timeout=120)
os.chdir(REPO_DIR)
subprocess.check_call(["git", "pull"], timeout=120)

KRONOS_DIR = os.path.join(REPO_DIR, "kronos_repo")
if not Path(KRONOS_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/shiyu-coder/Kronos.git",
                           KRONOS_DIR], timeout=120)

# -- Install deps --------------------------------------------------------------
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "yfinance", "torch", "transformers", "kagglehub"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])
print("Dependencies installed.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import kagglehub

CACHE_SRC = Path(REPO_DIR) / "data" / "forecast_cache"
DATASET_SLUG = f"kronos-forecast-cache-{YEAR}"

def _kaggle_user():
    return os.environ.get("KAGGLE_USERNAME", "")

def load_cache():
    if CACHE_SRC.exists() and any(CACHE_SRC.iterdir()):
        print(f"Cache already exists at {CACHE_SRC} -- skipping download")
        return
    user = _kaggle_user()
    if not user:
        print("No Kaggle user -- skipping cache load")
        return
    try:
        path = kagglehub.dataset_download(f"{user}/{DATASET_SLUG}")
        # Remove existing cache before overwriting
        if CACHE_SRC.exists():
            shutil.rmtree(CACHE_SRC)
        shutil.copytree(path, CACHE_SRC)
        count = len(list(CACHE_SRC.rglob("*.parquet")))
        print(f"Loaded {count} cached forecasts from Dataset: {DATASET_SLUG}")
    except Exception as e:
        print(f"No existing cache Dataset found: {e}")
        print("Starting fresh -- will create Dataset after precompute.")

def save_cache():
    if not CACHE_SRC.exists() or not any(CACHE_SRC.rglob("*.parquet")):
        print("No forecast cache to save -- skipping")
        return
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    user = _kaggle_user()
    dataset_ref = f"{user}/{DATASET_SLUG}"
    tmp = Path(f"/kaggle/working/{DATASET_SLUG}")
    tmp.mkdir(parents=True, exist_ok=True)
    zip_path = tmp / "forecast_cache.zip"
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", CACHE_SRC)
    size_mb = zip_path.stat().st_size / 1e6
    print(f"Cache zip: {size_mb:.0f} MB -- uploading as {dataset_ref} ...", flush=True)
    try:
        api.dataset_create_version(dataset_ref, str(zip_path), f"Cache update {time.ctime()}")
        print(f"Cache updated in Dataset: {dataset_ref}")
    except Exception:
        api.dataset_create(dataset_ref, upload_dir=str(tmp))
        print(f"Dataset created: {dataset_ref}")

if MODE in ("walkforward",):
    load_cache()

In [ ]:
if MODE in ("full", "precompute"):
    import time
    from kth.data.universe import UNIVERSE
    from kth.models.kronos_wrapper import KronosTH
    from kth.backtest.walkforward import BacktestConfig, precompute_forecasts
    from kth.data.loader import download_universe

    tickers = [x for x, _, _ in UNIVERSE['thai_equity']]
    end_date = "2026-05-30" if YEAR == "2026" else f"{YEAR}-12-31"

    # -- Download data ----------------------------------------------------------
    print("Downloading data...", flush=True)
    download_universe(tickers)

    # -- Load model -------------------------------------------------------------
    k = KronosTH.from_pretrained('NeoQuasar/Kronos-small', device='cuda')

    c = BacktestConfig(start_date=f'{YEAR}-01-01', end_date=end_date,
                       lookback=400, pred_len=20, n_samples=50,
                       position_sizing='equal', max_positions=5,
                       min_ticker_history=20)

    print(f'{YEAR}: {len(tickers)} tickers, n_samples=50', flush=True)
    t0 = time.time()
    precompute_forecasts(k, tickers, start_date=c.start_date, end_date=c.end_date,
                         pred_len=c.pred_len, n_samples=c.n_samples, lookback=c.lookback)
    print(f'PRECOMPUTE: {(time.time()-t0)/3600:.1f} hrs', flush=True)

    # -- Save cache as Kaggle Dataset -------------------------------------------
    save_cache()
else:
    print("Skipping precompute (MODE=walkforward)")

In [ ]:
if MODE in ("full", "walkforward"):
    from kth.data.universe import UNIVERSE
    from kth.models.kronos_wrapper import KronosTH
    from kth.backtest.walkforward import BacktestConfig, run_walkforward

    tickers = [x for x, _, _ in UNIVERSE['thai_equity']]
    end_date = "2026-05-30" if YEAR == "2026" else f"{YEAR}-12-31"

    k = KronosTH.from_pretrained('NeoQuasar/Kronos-small', device='cuda')
    c = BacktestConfig(start_date=f'{YEAR}-01-01', end_date=end_date,
                       lookback=400, pred_len=20, n_samples=50,
                       position_sizing='equal', max_positions=5,
                       min_ticker_history=20)

    print(f"Running walkforward for {YEAR}...", flush=True)
    r = run_walkforward(c, k, tickers)
    o = Path(f'data/backtest_results/thai_equity_{YEAR}_n50_v2')
    o.mkdir(parents=True, exist_ok=True)
    r.save(str(o))
    m = r.metrics
    ret = (r.equity_curve.iloc[-1] / r.equity_curve.iloc[0] - 1) if len(r.equity_curve) > 1 else 0.0
    print(f'DONE: Ret={ret:+.2%} Sharpe={m.get("sharpe",0):.2f} '
          f'MaxDD={m.get("max_drawdown",0):.2%} p={m.get("p_value",0):.3f}')
else:
    print("Skipping walkforward (MODE=precompute)")

In [ ]:
# -- Verify output ----------------------------------------------------------------
if MODE in ("full", "walkforward"):
    output_dir = f"data/backtest_results/thai_equity_{YEAR}_n50_v2"
    output_path = Path(output_dir)
    if output_path.exists():
        files = list(output_path.iterdir())
        print(f"Output dir: {output_dir}")
        print(f"Files: {len(files)}")
        for f in files[:10]:
            size_mb = f.stat().st_size / 1e6
            print(f"  {f.name} ({size_mb:.1f} MB)")
        if len(files) > 10:
            print(f"  ... and {len(files) - 10} more")
    else:
        print(f"No output dir yet -- run MODE=walkforward after precompute finishes")

# -- Cache status ---------------------------------------------------------------
if CACHE_SRC.exists():
    n_dates = len([d for d in CACHE_SRC.rglob("*.parquet")])
    print(f"Forecast cache: {n_dates} parquet files")
    print(f"To resume, MODE=precompute will skip cached dates.")

print(f"\nDone with {YEAR} MODE={MODE}.")

## Run Plan

### Session 1: `MODE = "precompute"`
- Download data + run precompute (~12 hrs, may timeout)
- Saves cache to Kaggle Dataset `kronos-forecast-cache-2023` before exiting
- **If timeout:** just restart with same YEAR, MODE=precompute -- resumes where it left off

### Session 2+: `MODE = "precompute"`
- Loads previous cache from Dataset, continues precomputing from where it stopped
- Repeat until you see "PRECOMPUTE: X.X hrs" (full year done)

### Final Session: `MODE = "walkforward"`
- Loads full cache from Dataset, runs walkforward (~1 hr)
- Saves results to `data/backtest_results/thai_equity_{YEAR}_n50_v2/`
- Download results before starting next year

### Troubleshooting
- **GPU not available:** Runtime -> Change -> GPU T4, Internet ON
- **Dataset already exists:** save_cache() uses `dataset_create_version` (update), not recreate
- **Cache too large:** If Dataset upload fails, manually zip and upload via Kaggle UI
- **yfinance blocked:** Switch dataset source